In [1]:
import os
import shutil
import getpass
#from google.colab import drive
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import torch
import torchvision
from torchvision import transforms
import torchvision.models


In [ ]:
!python --version

In [3]:
!nvidia-smi

Wed Mar 25 20:53:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-16GB           On  |   00000000:18:00.0 Off |                    0 |
| N/A   41C    P0             43W /  300W |       0MiB /  16384MiB |      0%   E. Process |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


In [8]:
torch.version.cuda

'12.8'

In [2]:
#!source plasticEnv_2/bin/activate

In [4]:
#NOTE on plasticEnv environment
#I created a python virtual environment and followed this tutorial: https://www.w3schools.com/python/python_virtualenv.asp
#This idea of creating a python virtual environment came from Connor Casey https://piazza.com/class/mkcrybcxyxq2ep/post/56



In [3]:
# mounting the drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

NameError: name 'drive' is not defined

In [ ]:
# setting up pathways
#drive_zip_folder = "/content/drive/MyDrive/Spring 2026/Deep Learning/Deep Learning Project/Wadaba Dataset/"
#local_path = "/content/wadaba_local"

#ben's directories
#drive_zip_folder = "/content/drive/MyDrive/Deep Learning Project/Wadaba Dataset/"
#local_path = "/content/drive/MyDrive/wadaba_local"

if os.path.exists("/content/drive/MyDrive/Deep Learning Project"):
  user = "Dani"
  drive_zip_folder = "/content/drive/MyDrive/Deep Learning Project/Wadaba Dataset/"

elif os.path.exists("/content/drive/MyDrive/Deep Learning Project"):
  user = "Ben"
  drive_zip_folder = "/content/drive/MyDrive/Deep Learning Project/Wadaba Dataset/"

else:
  user = "James"
  drive_zip_folder = "/content/drive/MyDrive/Deep Learning Project/Wadaba Dataset/" #put your path here

local_path = "/content/wadaba_local"

if os.path.exists(local_path) and len([f for f in os.listdir(local_path) if f.endswith(('.jpg', '.png'))]) > 4000:
    print(f"The images are already present in local SSD. Skipping unzip.")
else:
    print(f"Hi {user}!")
    pwd = getpass.getpass('Enter WaDaBa Zip Password: ')
    os.makedirs(local_path, exist_ok=True)

    # Extracting
    print("Extracting files... please wait.")
    !cd "{drive_zip_folder}" && unzip -P "{pwd}" -qo "*.zip" -d "{local_path}"

    print("Flattening subfolders...")
    for root, dirs, files in os.walk(local_path):
        if root == local_path: continue
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.move(os.path.join(root, file), os.path.join(local_path, file))

    # discarding empty folders
    for item in os.listdir(local_path):
        item_path = os.path.join(local_path, item)
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)

all_images = [f for f in os.listdir(local_path) if f.lower().endswith(('.jpg', '.png'))]

Hi Dani!
Enter WaDaBa Zip Password: ··········
Extracting files... please wait.
Flattening subfolders...


In [15]:
#class to go through files, splice labels and map to RIC for Pytorch tensors
class WadabaData(Dataset):
  def __init__(self, root_dir, transform=None): # initialize data loader
    self.root_dir = root_dir
    self.transform = transform

    self.image_files = [f for f in os.listdir(root_dir) if f.lower().endswith(('.jpg'))]
    self.label_map = {
        'a00': 0, # missing or unreadable
        'a01': 1, # PET
        'a02': 2, # PE-HD high density polyethylene
        'a03': 3, # PVC
        'a04': 4, # PE-LD
        'a05': 5, # PP
        'a06': 6, # PS
        'a07': 7  # other

    }

  #length for shuffling
  def __len__(self):
    return len(self.image_files)

  def __getitem__(self, idx):
    img_name = self.image_files[idx]
    img_path = os.path.join(self.root_dir, img_name)

    #load image
    image = Image.open(img_path).convert("RGB")

    #extract a0X
    label = 0 # default if not found
    for code, idx_val in self.label_map.items():
      if code in img_name:
        label = idx_val
        break
    if self.transform:
      image = self.transform(image)
    return image, torch.tensor(label)

In [16]:

#defining VIT-transforms
#VIT-16 is hardcoded for 224X224 so need to resize

#https://www.codegenes.net/blog/vit-pytorch/
vit_transforms = transforms.Compose([
    transforms.Resize((224,224)), #divides image into 16by16 patches
    transforms.RandomHorizontalFlip(), #adding data augmentation to make wadaba dataset "bigger"
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2,contrast=0.2), #changing lighting too if we want
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]) #VIT was previously trained on these numbers
])


In [17]:
#define vit transform
import torchvision.models
import torchvision.io
import numpy as np
import torch


cudaDevice = torch.device('cuda')

# [CITED DOCUMENTATION]
# https://docs.pytorch.org/vision/main/models/generated/torchvision.models.vit_b_16.html#torchvision.models.vit_b_16
# https://docs.pytorch.org/vision/stable/models.html
# https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html

#get pretrained weights
visionTransformerModelWeights = torchvision.models.ViT_B_16_Weights.DEFAULT

#get associated transforms
#visionTransformerModelTransforms = torchvision.models.ViT_B_16_Weights.DEFAULT.transforms

#define model
visionTransformerModel = torchvision.models.vit_b_16(weights=visionTransformerModelWeights)
#visionTransformerModel = torchvision.models.vit_b_16()


#got this idea from EC523 hmrk3 - adjusting output
visionTransformerModel.heads.head = torch.nn.Linear(in_features=768,out_features=8)




visionTransformerModel.to(cudaDevice)




VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [18]:
# Generating Train, Validation, and Test Sets

TOTAL_DATAPOINTS = 4000

#I think I've seen previous people cut up their datasets this way, maybe in EK381 and EC503 before?
TRAINING_SET_PERCENTAGE = .8
VALIDATION_SET_PERCENTAGE = .05
TESTING_SET_PERCENTAGE = .15

#[CITED DOCUMENTATION]
# https://numpy.org/doc/2.1/reference/random/generated/numpy.random.shuffle.html
# https://numpy.org/doc/2.1/reference/generated/numpy.arange.html
#had to look up how to create an array of integers and 'shuffle' the array using numpy
dataVals = np.arange(0,TOTAL_DATAPOINTS ,1)
np.random.shuffle(dataVals)

trainingSet = dataVals[0:int(TOTAL_DATAPOINTS*TRAINING_SET_PERCENTAGE)]
validationSet = dataVals[int(TOTAL_DATAPOINTS*TRAINING_SET_PERCENTAGE):int(TOTAL_DATAPOINTS*(TRAINING_SET_PERCENTAGE+VALIDATION_SET_PERCENTAGE))]
testingSet = dataVals[int(TOTAL_DATAPOINTS*(TRAINING_SET_PERCENTAGE+VALIDATION_SET_PERCENTAGE)):]


print(len(trainingSet))
print(len(validationSet))
print(len(testingSet))
print(trainingSet)


3200
200
600
[ 267 3524  633 ... 3128 2135 3049]


In [40]:
#NO BATCH IMPLEMENTATION




#local_path = "/content/drive/MyDrive/wadaba_local"
local_path = "/projectnb/ec523/projects/finalProjectPlastic/0_Data/wadaba_local"
dataset = WadabaData(local_path)
TOTAL_LABELS = 8

print(len(dataset))



for e in range(EPOCHS):
    
  print(e)

  for i in trainingSet:
    #print(i)

    #-------------------------- TRANSFORMATIONS

    #get first image and label
    imageData = dataset[i][0]
    imageLabel = dataset[i][1]

    #print(imageLabel)
    #print(imageData)

    # [CITED DOCUMENTATION]
    # https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
    # https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html

    #apply vit_transforms
    imageData = vit_transforms(imageData)

    '''
    #looked up how to change a PIL image to a numpy array: https://www.geeksforgeeks.org/python/how-to-convert-images-to-numpy-array/
    imageData = np.array(imageData)
    height = image.shape[0]
    width = image.shape[1]
    layers = image.shape[2]
    image = image.reshape(layers, height,width)
    '''

    #convert to tensor
    imageData = torch.Tensor(imageData).cuda()

    #additional transformation
    imageData = imageData.unsqueeze(0)
    imageData = imageData.float()
    
    print(torch.Tensor.type(imageData))
    


    #-------------------------- FORWARD/BACKWARD PASSES


    #print('output')
    output = visionTransformerModel(imageData)
    prediction = torch.nn.functional.softmax(output)

    #[CITED DOCUMENTATION]
    # https://piazza.com/class/mkcrybcxyxq2ep/post/21

    #one hot label created similar to how Weicheng showed us
    imageLabelOneHot = torch.Tensor(np.eye(TOTAL_LABELS)[imageLabel]).unsqueeze(0)

    #print(imageLabelOneHot.shape)
    #print(prediction.shape)


    loss = crossEntropyLoss(imageLabelOneHot,prediction)
    print(loss)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()















4000
0
torch.cuda.FloatTensor


/scratch/3960036.1.academic-gpu/ipykernel_3962631/2356411313.py:60: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  prediction = torch.nn.functional.softmax(output)


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [115]:
local_path = "/projectnb/ec523/projects/finalProjectPlastic/0_Data/wadaba_local"
dataset = WadabaData(local_path)

#dSet = torch.utils.data.DataLoader(dataset)



In [ ]:
#IMPLEMENTED WITH BATCH


#declare hyperparameters
LEARNING_RATE = .01
MOMENTUM = .8
EPOCHS = 10


#declare loss function
crossEntropyLoss = torch.nn.CrossEntropyLoss()

#declare optimizer
optimizer = torch.optim.SGD(visionTransformerModel.parameters(),lr = LEARNING_RATE, momentum = MOMENTUM)


#local_path = "/content/drive/MyDrive/wadaba_local"
local_path = "/projectnb/ec523/projects/finalProjectPlastic/0_Data/wadaba_local"
dataset = WadabaData(local_path)
TOTAL_LABELS = 8
BATCH_SIZE = 7




#images that cause errors in the training process
bad_images = [1240,3399]

visionTransformerModel.train()



for e in range(EPOCHS):
    
  print("EPOCH: ----------------------------- ", e,"out of",EPOCHS)

  for i in range(0,len(trainingSet),BATCH_SIZE):
        
        
        #print("Epoch Progress: ",i/len(trainingSet))
        
        batch = []
        batch_labels = []
        for j in range(0,BATCH_SIZE):

            #-------------------------- TRANSFORMATIONS
            
            #[CITED DOCUMENTATION]
            #looked up how to implement a "try" and "Except" workflow
            #https://www.w3schools.com/python/python_try_except.asp
            try:
                #get first image and label
                imageData = dataset[trainingSet[i+j]][0]
                imageLabel = dataset[trainingSet[i+j]][1]



                #print("Training Image Index: ",trainingSet[i+j])
                
                #print(imageData)

                # [CITED DOCUMENTATION]
                # https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
                # https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html

                #apply vit_transforms
                imageData = vit_transforms(imageData)

                '''
                #looked up how to change a PIL image to a numpy array: https://www.geeksforgeeks.org/python/how-to-convert-images-to-numpy-array/
                imageData = np.array(imageData)
                height = image.shape[0]
                width = image.shape[1]
                layers = image.shape[2]
                image = image.reshape(layers, height,width)
                '''

                # [CITED DOCUMENATION]
                #needed to convert tensor to numpy
                #https://docs.pytorch.org/docs/stable/generated/torch.Tensor.numpy.html
                batch.append(imageData.numpy())
                batch_labels.append(imageLabel.numpy())
            
            except:
                None

        batch = np.array(batch)
        batch_labels = np.array(batch_labels)
        
        #print(batch)
        #print(batch_labels)

        #convert to tensor
        imageData = torch.Tensor(batch).to(cudaDevice)
        
        

        #additional transformation
        #imageData = imageData.unsqueeze(0)
        imageData = imageData.float()
        
        
        #print(torch.Tensor.type(imageData))
        
        #print(imageData)
        #print(imageData.shape)
        
        
        
        
        
        


    #-------------------------- FORWARD/BACKWARD PASSES


        #print('output')
        output = visionTransformerModel(imageData)
        prediction = torch.nn.functional.softmax(output)
        
        #print(prediction)

        #[CITED DOCUMENTATION]
        # https://piazza.com/class/mkcrybcxyxq2ep/post/21

        #one hot label created similar to how Weicheng showed us
        imageLabelOneHot = torch.Tensor(np.eye(TOTAL_LABELS)[batch_labels]).to(cudaDevice)

        #print(imageLabelOneHot.shape)
        #print(prediction.shape)
        
        #print(imageLabelOneHot)


        loss = crossEntropyLoss(imageLabelOneHot,prediction)
        #print(loss)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()










correct = 0
visionTransformerModel.eval()

for i in validationSet:

    try: 
        #-------------------------- TRANSFORMATIONS

        #get first image and label
        imageData = dataset[i][0]
        imageLabel = dataset[i][1]

        #print(imageLabel)
        #print(imageData)

        # [CITED DOCUMENTATION]
        # https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
        # https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html

        #apply vit_transforms
        imageData = vit_transforms(imageData)

        '''
        #looked up how to change a PIL image to a numpy array: https://www.geeksforgeeks.org/python/how-to-convert-images-to-numpy-array/
        imageData = np.array(imageData)
        height = image.shape[0]
        width = image.shape[1]
        layers = image.shape[2]
        image = image.reshape(layers, height,width)
        '''

        #convert to tensor
        imageData = torch.Tensor(imageData).to(cudaDevice)

        #additional transformation
        imageData = imageData.unsqueeze(0)
        imageData = imageData.float()

        #print(torch.Tensor.type(imageData))



        #-------------------------- FORWARD/BACKWARD PASSES


        #print('output')
        output = visionTransformerModel(imageData)
        prediction = torch.nn.functional.softmax(output)

        #[CITED DOCUMENTATION]
        # https://piazza.com/class/mkcrybcxyxq2ep/post/21


        if imageLabel.item() == torch.argmax(prediction).item():
            correct += 1

        #print(imageLabel.item(),torch.argmax(prediction).item())
    
    except:
        None

        
print(correct/len(testingSet))








EPOCH: -----------------------------  0 out of 10


/scratch/3964677.1.academic-gpu/ipykernel_3978704/1186002301.py:120: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  prediction = torch.nn.functional.softmax(output)


EPOCH: -----------------------------  1 out of 10
EPOCH: -----------------------------  2 out of 10


In [19]:
#TEST PROCEDURE


correct = 0
visionTransformerModel.eval()

for i in testingSet:
    
    try:

        #-------------------------- TRANSFORMATIONS

        #get first image and label
        imageData = dataset[i][0]
        imageLabel = dataset[i][1]

        #print(imageLabel)
        #print(imageData)

        # [CITED DOCUMENTATION]
        # https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
        # https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html

        #apply vit_transforms
        imageData = vit_transforms(imageData)

        '''
        #looked up how to change a PIL image to a numpy array: https://www.geeksforgeeks.org/python/how-to-convert-images-to-numpy-array/
        imageData = np.array(imageData)
        height = image.shape[0]
        width = image.shape[1]
        layers = image.shape[2]
        image = image.reshape(layers, height,width)
        '''

        #convert to tensor
        imageData = torch.Tensor(imageData).to(cudaDevice)

        #additional transformation
        imageData = imageData.unsqueeze(0)
        imageData = imageData.float()

        #print(torch.Tensor.type(imageData))



        #-------------------------- FORWARD/BACKWARD PASSES


        #print('output')
        output = visionTransformerModel(imageData)
        prediction = torch.nn.functional.softmax(output)

        #[CITED DOCUMENTATION]
        # https://piazza.com/class/mkcrybcxyxq2ep/post/21


        if imageLabel.item() == torch.argmax(prediction).item():
            correct += 1

        print(imageLabel.item(),torch.argmax(prediction).item())
    
    except:
        None

        
print(correct/len(testingSet))





/scratch/3964677.1.academic-gpu/ipykernel_3978704/2890176894.py:52: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  prediction = torch.nn.functional.softmax(output)


1 7
7 7
1 5
2 1
5 7
6 6
1 1
2 6
2 4
6 1
2 7
1 7
1 1
1 7
1 7
6 7
1 7
2 2
2 1
6 7
5 2
6 2
6 2
1 3
5 7
5 7
6 3
2 7
1 7
1 7
1 1
2 7
1 7
2 7
6 3
5 1
1 7
2 1
1 7
1 7
6 7
6 2
5 7
1 7
1 1
1 7
7 7
6 2
5 1
6 1
1 7
1 7
1 7
5 1
5 7
1 1
5 3
1 6
1 7
1 7
1 7
1 4
5 7
5 6
5 7
7 7
1 7
2 1
1 7
5 7
2 7
5 7
2 7
1 7
1 7
5 7
1 7
6 5
1 7
1 7
2 7
1 7
1 7
5 7
5 7
2 1
1 1
1 7
1 0
1 7
2 3
5 7
5 7
2 3
1 7
2 6
2 2
1 7
1 7
1 3
1 7
1 7
1 7
2 7
6 7
1 7
1 7
1 4
1 7
2 7
1 7
1 7
5 1
5 7
6 3
2 7
1 7
1 7
1 7
6 7
5 7
1 7
1 7
2 2
5 0
1 7
2 7
1 7
1 5
5 1
5 3
5 1
5 7
6 7
1 7
1 7
1 7
1 7
1 7
5 7
1 7
2 5
5 5
1 7
5 1
1 7
5 7
2 1
1 7
1 7
6 3
1 1
5 5
5 6
1 7
2 7
1 7
1 1
2 7
2 6
2 4
2 3
1 7
1 1
6 7
1 7
1 5
1 7
2 7
1 7
1 7
2 7
1 4
1 2
5 1
5 7
1 7
1 7
5 7
2 6
1 7
1 3
1 7
2 6
5 6
1 7
7 7
1 3
1 4
1 1
1 7
2 5
6 7
1 7
1 7
6 2
6 1
5 7
6 1
1 7
5 7
2 7
6 1
1 7
1 7
2 2
1 3
1 7
2 7
5 7
2 6
1 7
6 7
1 1
6 1
5 7
5 7
1 1
5 7
5 7
1 7
1 7
6 0
1 7
5 7
1 3
1 3
1 7
6 2
6 4
5 7
1 7
6 7
1 7
5 7
1 7
1 3
1 7
1 4
1 6
1 3
5 7
1 7
1 7
1 2
5 7
5 7
6 7
6 7
1 5


In [34]:
!nvidia-smi

Wed Mar 25 19:07:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-16GB           On  |   00000000:18:00.0 Off |                    0 |
| N/A   44C    P0             62W /  300W |    1062MiB /  16384MiB |      0%   E. Process |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [122]:
#torch.cuda.is_available()

print(torch.version.cuda)

12.8


In [ ]:
#OLD CODE: test image runthrough

test_image = "/content/drive/MyDrive/wadaba_local/0099_a02b01c1d1e0f1g0h4.jpg"

testTensor = torchvision.io.read_image(test_image)
testTensor = testTensor.unsqueeze(0)
testTensor = testTensor.float()


# [CITED DOCUMENTATION]
# https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
# https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html

#declare transforms
changeSize = torchvision.transforms.Resize((224,224))
normValues = torchvision.transforms.Normalize((120,120,120),(1,1,1))

testTensor = changeSize(testTensor)
testTensor = normValues(testTensor)

print(testTensor)
print(testTensor.shape)

print('output')
output = visionTransformerModel(testTensor)
prediction = torch.nn.functional.softmax(output)

testLabel = 4
total_labels = 1000

#[CITED DOCUMENTATION]
# https://piazza.com/class/mkcrybcxyxq2ep/post/21

#one hot label created similar to how Weicheng showed us
testLabelOneHot = torch.Tensor(np.eye(total_labels)[testLabel]).unsqueeze(0)

print(testLabelOneHot.shape)
print(prediction.shape)

loss = crossEntropyLoss(testLabelOneHot,prediction)

optimizer.zero_grad()
loss.backward()
optimizer.step()


#print(torch.argmax(prediction))


